## Install Dependencies

In [ ]:
!pip install langchain langchain-groq langchain-huggingface langchain-chroma sentence-transformers

## Imports

In [ ]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain.retrievers.multi_query import MultiQueryRetriever
import logging

## Setup Components

In [ ]:
os.environ['GROQ_API_KEY'] = 'gsk_YOUR_API_KEY_HERE'
llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0)

with open("dummy_data2.txt", "w") as f:
    f.write("Project Alpha aims to reduce server latency by 50%. It relies on Kubernetes.")

chunks = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20).split_documents(TextLoader("dummy_data2.txt").load())
vectorstore = Chroma.from_documents(chunks, HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"))

retriever_from_llm = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(), llm=llm
)

## Advanced Retrieval Function

In [ ]:
def query_advanced(query):
    # Will generate multiple query variations under the hood to ensure recall
    return retriever_from_llm.invoke(query)

## Test Multi-Query Retrieval

In [ ]:
docs = query_advanced("Tell me about the goals of the Alpha project and its tech stack.")